In [ ]:
# Visualization Summary
# Load trip data from Google Cloud Storage (GCS).
# Show basic stats on trip and tip info.
# Create visualizations:
####Tip bucket counts by pickup location
#### Heatmap of pickup/drop-off pairs
#### Tip amount vs trip duration
#### Avg tip by time of day and weekend
# Load trained logistic regression model.
####Extract and plot top feature importances.
####Evaluate model with ROC curve and AUC.
# Save all plots and upload them to GCS.


# PySpark Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnan, when, count, udf, to_date, year, month, date_format, size, split, avg
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.linalg import DenseVector

# Visualization Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc
import io

# Google Cloud Storage
from google.cloud import storage

# Initialize Spark
spark = SparkSession.builder.master("local[3]").appName("Visualizations").getOrCreate()
sc = spark.sparkContext

# GCS Paths
BUCKET_NAME    = "my-bigdata-project-kc"
MODEL_PATH     = f"gs://{BUCKET_NAME}/models/logistic_model"
ASSEMBLED_PATH = f"gs://{BUCKET_NAME}/trusted/assembled_features"
OUTPUT_PATH    = f"gs://{BUCKET_NAME}/models/filtered_feature_importance"
FIGURE_FOLDER  = "figures"
FIGURE_PATH    = f"gs://{BUCKET_NAME}/{FIGURE_FOLDER}"

#Save and Upload Plots to GCS Function

def save_and_upload_plot(fig, filename, bucket_name, figure_folder):

    # Create in-memory image buffer
    img_data = io.BytesIO()

    # Save the figure to the buffer BEFORE closing or displaying
    fig.savefig(img_data, format='png', bbox_inches='tight')
    img_data.seek(0)

    # Upload image to GCS
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(f"{figure_folder}/{filename}")
    blob.upload_from_file(img_data, content_type='image/png')

    print(f"Plot uploaded to: gs://{bucket_name}/{figure_folder}/{filename}")

    # Display the image
    plt.show()

    # Close
    plt.close(fig)

    # Load Assembled Features SDF
EXCLUDED_KEYWORDS = ["trusted/part"]

def read_all_parquet_files():
    storage_client = storage.Client()
    blobs = storage_client.list_blobs(BUCKET_NAME, prefix="trusted/")

    parquet_blobs = [
        blob for blob in blobs
        if blob.name.endswith('.parquet') and not any(keyword in blob.name for keyword in EXCLUDED_KEYWORDS)
    ]

    df_list = []
    for blob in parquet_blobs:
        cleaned_filename = f"gs://{BUCKET_NAME}/{blob.name}"
        sdf_temp = spark.read.parquet(cleaned_filename)
        df_list.append(sdf_temp)
        print(f"Successfully added: {blob.name}")

    if df_list:
        sdf = df_list[0]
        for df in df_list[1:]:
            sdf = sdf.union(df)
    else:
        sdf = spark.createDataFrame(spark.sparkContext.emptyRDD(), schema=None)
        print("No valid Parquet files found in the folder.")

    return sdf
 print(f" All files successfully added.")

sdf_filled = read_all_parquet_files()

# Summary Stats
sdf_filled.count()
sdf_filled.select("trip_miles","trip_duration", "trip_weekend", "pickup_time_binary","tips").summary("count", "min", "max", "mean").show()

# Tip Bucket Distribution Plot
top_pickups = (
    sdf_filled.groupBy('PULocationID')
              .agg({'tips': 'sum'})
              .withColumnRenamed('sum(tips)', 'total_tip_amount')
              .orderBy('total_tip_amount', ascending=False)
              .limit(100000)
)

sdf_top_pickups = sdf_filled.join(top_pickups, on='PULocationID', how='inner')
df = sdf_top_pickups.select('PULocationID', 'tip_bucket').limit(50000).toPandas()

# Create figure object and plot
sns.set_style("whitegrid")
fig = plt.figure(figsize=(12, 8))  # Capture the figure object
sns.countplot(x='tip_bucket', data=df, palette='Set2')
plt.ylabel('Count')
plt.title('Tip Bucket Distribution for Top Pickup Locations by Total Tips')
plt.tight_layout()
#plt.show()

# Save and upload the plot using the utility function
save_and_upload_plot(fig, "top_pickups_by_total_tips_no_weekend.png", BUCKET_NAME, FIGURE_FOLDER)

# Heatmap of Top Pickup vs Drop-off Pairs
top5_pairs = sdf_filled.groupBy("PULocationID", "DOLocationID") \
                       .count().orderBy(col("count").desc()).limit(5).toPandas()

pivot_df = top5_pairs.pivot(index="PULocationID", columns="DOLocationID", values="count").fillna(0)
fig1 =plt.figure(figsize=(8, 6))
sns.heatmap(pivot_df, annot=True, fmt="g", cmap="YlGnBu")
plt.title("Top 5 Pickup vs Drop-off Location Pairs")
plt.tight_layout()
plt.show()
save_and_upload_plot(fig1, "pickup_dropoff_heatmap_top5.png", BUCKET_NAME, FIGURE_FOLDER)


# Trip Duration vs Tip Scatter Plot
df_scatter = sdf_filled.select("trip_duration", "tips").dropna().limit(50000).toPandas()
fig2 =plt.figure(figsize=(8, 6))
sns.scatterplot(x="trip_duration", y="tips", data=df_scatter, alpha=0.5)
plt.title("Trip Duration vs Tip Amount")
plt.xlabel("Trip Duration (minutes)")
plt.ylabel("Tip Amount ($)")
plt.tight_layout()
plt.show()
save_and_upload_plot(fig2, "trip_duration_vs_tip.png", BUCKET_NAME, FIGURE_FOLDER)


# Average Tip by Time of Day and Weekend
df_avg_tip = sdf_filled.groupBy("pickup_time_binary", "trip_weekend") \
                       .agg(avg("tips").alias("avg_tip")).limit(50000)

df_avg_tip_pd = df_avg_tip.toPandas()
fig3 =plt.figure(figsize=(8, 6))
sns.barplot(x="pickup_time_binary", y="avg_tip", hue="trip_weekend", data=df_avg_tip_pd)
plt.title("Average Tip by Time of Day and Weekend")
plt.xlabel("Time of Day (0 = AM, 1 = PM)")
plt.ylabel("Average Tip ($)")
plt.tight_layout()
plt.show()
save_and_upload_plot(fig3, "avg_tip_by_time_and_weekend.png", BUCKET_NAME, FIGURE_FOLDER)

# Load Logistic Regression Model
try:
    lr_model = LogisticRegressionModel.load(MODEL_PATH)
    print("Model loaded from:", MODEL_PATH)
except Exception as e:
    print("Error loading model:", e)
    raise

#Feature Importance

coeff_array = lr_model.coefficients.toArray()
input_features = ["trip_miles", "trip_duration", "trip_weekend", "pickup_time_binary"]
pu_size, do_size = 50, 50
feature_names = input_features + [f"PULocationID_vec_{i}" for i in range(pu_size)] + [f"DOLocationID_vec_{i}" for i in range(do_size)]

feat_imp = list(zip(feature_names, coeff_array))
sorted_imp = sorted(feat_imp, key=lambda x: abs(x[1]), reverse=True)
non_loc = [f for f in sorted_imp if not f[0].startswith(("PULocationID_vec", "DOLocationID_vec"))]
pu_top5 = [f for f in sorted_imp if f[0].startswith("PULocationID_vec")][:5]
do_top5 = [f for f in sorted_imp if f[0].startswith("DOLocationID_vec")][:5]
final_feats = non_loc + pu_top5 + do_top5

print("\nTop Feature Importances:")
for name, coef in final_feats:
    print(f"{name:<30} {coef:+.4f}")

final_df = pd.DataFrame(final_feats, columns=["feature", "coefficient"])
sdf_final = spark.createDataFrame(final_df)
sdf_final.coalesce(1).write.mode("overwrite").option("header", "true").csv(OUTPUT_PATH)
print(f"\nFiltered importances saved to: {OUTPUT_PATH}")

#Barplot of Feature Importances

plot_df = final_df[final_df['coefficient'].abs() > 0.01]

fig5 = plt.figure(figsize=(12, 6))
barplot = sns.barplot(data=plot_df, x='coefficient', y='feature')
plt.title("Important Features with |Importance| > 0.01")
plt.xlabel("Importance Score")
plt.ylabel("Feature Name")


save_and_upload_plot(fig5, "feature_importance_barplot.png", BUCKET_NAME, FIGURE_FOLDER)
plt.close()

# Boxplot of Importance Distribution (if helpful)
plot_df = final_df[final_df['coefficient'].abs() > 0.01]

fig6=plt.figure(figsize=(10, 6))
boxplot = sns.boxplot(x=plot_df['coefficient'], orient='h')
plt.title("Distribution of Feature Importances > 0.01")
plt.xlabel("Importance Score")


save_and_upload_plot(fig6, "feature_importance_boxplot.png", BUCKET_NAME, FIGURE_FOLDER)
plt.close()

#ROC Curve

# Limit the dataset to the first 100,000 records

sample_df = sdf_filled.limit(100000)

# Generate predictions on the limited data
preds = lr_model.transform(sample_df)

# Extract probability and label, convert to Pandas
pdf = (
    preds.select("probability", "label")
         .rdd
         .map(lambda r: (float(r.probability[1]), float(r.label)))
         .toDF(["prob", "label"])
         .toPandas()
)

# Compute ROC curve and AUC
fpr, tpr, _ = roc_curve(pdf.label, pdf.prob)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
fig4 = plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], lw=2, linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Sample Data - 100,000 Records)")
plt.legend(loc="lower right")
plt.tight_layout()

# Save and upload using the provided function
save_and_upload_plot(fig4, "roc_curve.png", BUCKET_NAME, FIGURE_FOLDER)





